In [4]:
import numpy as np
from collections import defaultdict
 
FP_FRAME_SIZE   = 4096   # STFT window (samples)
FP_HOP_SIZE     = 2048   # STFT hop (samples)  -> 50% overlap
FP_FAN_OUT      = 5      # how many target peaks pair with each anchor
FP_TARGET_DT    = 100    # max frame-distance between anchor and target peak
FP_MIN_DT       = 1      # min frame-distance (avoid pairing a peak with itself)
FP_NEIGHBORHOOD = 20     # local-maximum neighborhood (freq bins x time frames)
FP_MIN_MATCHES  = 8      # min aligned-hash votes required to call a match
 
fingerprint_db = defaultdict(list)     # hash -> [(track_id, anchor_frame), ...]
 
 
def _spectrogram(samples, sr):
    """Magnitude spectrogram via manual STFT (Hann window)."""
    window = np.hanning(FP_FRAME_SIZE)
    n_frames = 1 + (len(samples) - FP_FRAME_SIZE) // FP_HOP_SIZE
    if n_frames <= 0:
        return np.empty((0, FP_FRAME_SIZE // 2 + 1))
    spec = np.empty((n_frames, FP_FRAME_SIZE // 2 + 1), dtype=np.float64)
    for i in range(n_frames):
        start = i * FP_HOP_SIZE
        frame = samples[start:start + FP_FRAME_SIZE] * window
        spec[i] = np.abs(np.fft.rfft(frame))
    return spec
 
 
def _find_peaks(spec):
    """Local maxima in the time-frequency plane -> list of (frame_idx, freq_bin)."""
    from scipy.ndimage import maximum_filter
    if spec.size == 0:
        return []
    local_max = maximum_filter(spec, size=(FP_NEIGHBORHOOD, FP_NEIGHBORHOOD)) == spec
    threshold = np.percentile(spec, 90)   # keep only strong peaks
    mask = local_max & (spec > threshold)
    frames, freqs = np.nonzero(mask)
    return list(zip(frames.tolist(), freqs.tolist()))
 
 
def _peaks_to_hashes(peaks):
    """Pair each anchor peak with nearby peaks -> (hash, anchor_frame)."""
    peaks = sorted(peaks)  # sort by frame index
    hashes = []
    n = len(peaks)
    for i in range(n):
        f1, freq1 = peaks[i]
        paired = 0
        for j in range(i + 1, n):
            f2, freq2 = peaks[j]
            dt = f2 - f1
            if dt < FP_MIN_DT:
                continue
            if dt > FP_TARGET_DT:
                break
            # pack (freq1, freq2, dt) into a single integer hash
            h = (int(freq1) << 22) | (int(freq2) << 11) | int(dt)
            hashes.append((h, f1))
            paired += 1
            if paired >= FP_FAN_OUT:
                break
    return hashes
 
 
def fingerprint_register(samples, sr, track_id):
    """Compute hashes for a reference track and add them to the in-memory DB."""
    spec = _spectrogram(samples, sr)
    peaks = _find_peaks(spec)
    for h, anchor_frame in _peaks_to_hashes(peaks):
        fingerprint_db[h].append((track_id, anchor_frame))
 
 
def fingerprint_recognize(samples, sr):
    """
    Attempt to identify a query clip against the registered DB.
    Returns (track_id_or_None, confidence_vote_count).
    """
    spec = _spectrogram(samples, sr)
    peaks = _find_peaks(spec)
    query_hashes = _peaks_to_hashes(peaks)
 
    # For each candidate track, tally votes for each (db_frame - query_frame) offset.
    # A real match produces many hashes aligned to the SAME offset; noise doesn't.
    offset_votes = defaultdict(lambda: defaultdict(int))
    for h, q_frame in query_hashes:
        for track_id, db_frame in fingerprint_db.get(h, []):
            offset = db_frame - q_frame
            offset_votes[track_id][offset] += 1
 
    best_track, best_count = None, 0
    for track_id, offsets in offset_votes.items():
        top_offset_count = max(offsets.values())
        if top_offset_count > best_count:
            best_track, best_count = track_id, top_offset_count
 
    if best_count >= FP_MIN_MATCHES:
        return best_track, best_count
    return None, best_count
 
 
# ============================================================
# CELL 4 — WATERMARK DETECTION (pairs with your embed() in Cell 2)
# ============================================================
 
ID_LENGTH_CHARS = 8   # convention: all custom_artist_id values are 8 chars
ID_BITS = ID_LENGTH_CHARS * 8
 
 
def binary_to_text(bits):
    chars = [
        chr(int("".join(str(b) for b in bits[i:i + 8]), 2))
        for i in range(0, len(bits) - 7, 8)
    ]
    return "".join(chars)
 
 
def watermark_decode_bits(samples, frame_length=None, overlap=None,
                           rep_code=None, num_reps=None, prs_seq=None):
    """
    Blind-decode bits from a (possibly degraded) watermarked signal by
    correlating each frame against the same PRS sequence used at embed time.
    Mirrors the frame stepping logic of watermark_audio() in Cell 2.
    """
    frame_length = frame_length or FRAME_LENGTH
    overlap = overlap if overlap is not None else OVERLAP
    rep_code = REP_CODE if rep_code is None else rep_code
    num_reps = NUM_REPS if num_reps is None else num_reps
    prs_seq = prs if prs_seq is None else prs_seq
 
    frame_shift = int(frame_length * (1 - overlap))
    n_frames_needed = ID_BITS * num_reps if rep_code else ID_BITS
    max_frames = (len(samples) - frame_length) // frame_shift
    n_frames = min(n_frames_needed, max_frames)
    if n_frames <= 0:
        return None
 
    raw_bits = np.empty(n_frames, dtype=np.int8)
    pointer = 0
    for i in range(n_frames):
        frame = samples[pointer:pointer + frame_length]
        correlation = np.dot(frame - frame.mean(), prs_seq[:len(frame)])
        raw_bits[i] = 1 if correlation > 0 else 0
        pointer += frame_shift
 
    if rep_code:
        usable = (len(raw_bits) // num_reps) * num_reps
        raw_bits = raw_bits[:usable].reshape(-1, num_reps)
        decoded_bits = (raw_bits.mean(axis=1) >= 0.5).astype(np.int8)
    else:
        decoded_bits = raw_bits
 
    return decoded_bits[:ID_BITS]
 
 
def watermark_detect(samples, expected_text=None, registered_ids=None):
    """
    Decode the watermark and identify the track.
    - If expected_text is given (unit-test mode): also returns BER against
      the known ground-truth bit string.
    - If registered_ids is given (stream mode, no known answer): matches the
      decoded ID against the registry, allowing 0 or 1 bit-level ID errors.
    Returns dict(decoded_text, identified_id_or_None, ber_or_None).
    """
    decoded_bits = watermark_decode_bits(samples)
    if decoded_bits is None or len(decoded_bits) < ID_BITS:
        return {"decoded_text": None, "identified_id": None, "ber": None}
 
    decoded_text = binary_to_text(decoded_bits)
 
    ber = None
    if expected_text is not None:
        expected_bits = np.array(
            [int(b) for b in text_to_binary(expected_text)[:ID_BITS]], dtype=np.int8
        )
        ber = float(np.mean(decoded_bits != expected_bits))
 
    identified_id = None
    if registered_ids is not None:
        if decoded_text in registered_ids:
            identified_id = decoded_text
        else:
            # allow near-matches (<=1 character different) for robustness
            for candidate in registered_ids:
                if sum(a != b for a, b in zip(decoded_text, candidate)) <= 1:
                    identified_id = candidate
                    break
    elif expected_text is not None and decoded_text == expected_text:
        identified_id = decoded_text
 
    return {"decoded_text": decoded_text, "identified_id": identified_id, "ber": ber}
 
 
# ============================================================
# CELL 5 — DEGRADATION UTILITIES (five test conditions)
# ============================================================
 
import subprocess, tempfile, os as _os
 
DEGRADATION_CONDITIONS = ["clean", "mp3_64k", "mp3_32k", "noise_snr20", "noise_snr10"]
 
 
def degrade_samples(samples, sr, condition):
    """Returns a degraded copy of an int16 mono sample array."""
    if condition == "clean":
        return samples.copy()
 
    if condition.startswith("mp3_"):
        bitrate = condition.split("_")[1]
        with tempfile.TemporaryDirectory() as td:
            wav_in = _os.path.join(td, "in.wav")
            mp3_out = _os.path.join(td, "out.mp3")
            wav_out = _os.path.join(td, "out.wav")
            wavfile.write(wav_in, sr, samples.astype(np.int16))
            subprocess.run(["ffmpeg", "-y", "-i", wav_in, "-b:a", bitrate, mp3_out],
                            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.run(["ffmpeg", "-y", "-i", mp3_out, wav_out],
                            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            out_sr, out_samples = wavfile.read(wav_out)
            return out_samples.astype(np.float64)
 
    if condition.startswith("noise_snr"):
        target_snr_db = float(condition.replace("noise_snr", ""))
        signal_power = np.mean(samples.astype(np.float64) ** 2)
        noise_power = signal_power / (10 ** (target_snr_db / 10))
        noise = rng.normal(0, np.sqrt(noise_power), samples.shape)
        return np.clip(samples + noise, -32768, 32767)
 
    raise ValueError(f"Unknown condition: {condition}")
 
 
# ============================================================
# CELL 6 — UNIT-LEVEL VALIDATION (per-file, both engines, 5 conditions)
# ============================================================
 
def load_mono_samples(path, target_sr=None):
    audio = AudioSegment.from_file(path).set_channels(1)
    if target_sr:
        audio = audio.set_frame_rate(target_sr)
    samples = np.array(audio.get_array_of_samples(), dtype=np.float64)
    return samples, audio.frame_rate
 
 
def run_unit_level_tests(metadata, conditions=DEGRADATION_CONDITIONS):
    """
    For each track: register its fingerprint, then re-recognize it under
    each condition (should identify itself). Separately, watermark it (using
    your Cell 2 watermark_audio), then re-detect it under each condition.
    """
    unit_results = []
    fingerprint_db.clear()
 
    # --- Step 1: register clean fingerprints for every track once ---
    track_samples = {}
    for record in metadata:
        track_id = str(record["custom_track_id"])
        samples, sr = load_mono_samples(record["file"])
        track_samples[track_id] = (samples, sr)
        fingerprint_register(samples, sr, track_id)
 
    registered_ids = {str(r["custom_artist_id"]) for r in metadata}
 
    # --- Step 2: test each track under each condition ---
    for record in metadata:
        track_id = str(record["custom_track_id"])
        artist_id = str(record["custom_artist_id"])
        samples, sr = track_samples[track_id]
 
        for condition in conditions:
            degraded = degrade_samples(samples, sr, condition)
 
            t0 = time.perf_counter()
            fp_match, fp_votes = fingerprint_recognize(degraded, sr)
            fp_latency = time.perf_counter() - t0
            fp_correct = (fp_match == track_id)
 
            wm_path = _os.path.join(OUTPUT_DIR, f"{track_id}_watermarked.wav")
            if _os.path.isfile(wm_path):
                wm_sr, wm_samples = wavfile.read(wm_path)
                wm_degraded = degrade_samples(wm_samples.astype(np.float64), wm_sr, condition)
                wm_result = watermark_detect(wm_degraded, expected_text=artist_id,
                                              registered_ids=registered_ids)
            else:
                wm_result = {"decoded_text": None, "identified_id": None, "ber": None}
 
            unit_results.append({
                "track_id": track_id, "artist_id": artist_id, "condition": condition,
                "fp_correct": fp_correct, "fp_votes": fp_votes, "fp_latency_s": fp_latency,
                "wm_identified": wm_result["identified_id"] == artist_id,
                "wm_ber": wm_result["ber"],
            })
 
    return unit_results
 
 
# ============================================================
# CELL 7 — BUILD SIMULATED BROADCAST (stream-level ground truth)
# ============================================================
 
import time
 
CLIP_SECONDS = 28
 
def build_simulated_broadcast(metadata, target_sr=44100):
    """
    Concatenates a fixed-length clip of each track's WATERMARKED audio
    (so a single broadcast tests both engines) into one continuous stream,
    recording ground-truth start/end timestamps per Section 3.6.4.
    """
    broadcast_chunks = []
    ground_truth = []
    cursor_samples = 0
 
    for record in metadata:
        track_id = str(record["custom_track_id"])
        artist_id = str(record["custom_artist_id"])
        wm_path = _os.path.join(OUTPUT_DIR, f"{track_id}_watermarked.wav")
        source_path = wm_path if _os.path.isfile(wm_path) else record["file"]
 
        samples, sr = load_mono_samples(source_path, target_sr=target_sr)
        clip_len = int(CLIP_SECONDS * target_sr)
        clip = samples[:clip_len]
        if len(clip) < clip_len:
            clip = np.pad(clip, (0, clip_len - len(clip)))
 
        broadcast_chunks.append(clip)
        duration = len(clip) / target_sr
        ground_truth.append({
            "track_id": track_id, "artist_id": artist_id,
            "start_sec": cursor_samples / target_sr,
            "end_sec": cursor_samples / target_sr + duration,
        })
        cursor_samples += len(clip)
 
    broadcast = np.concatenate(broadcast_chunks)
    return broadcast, target_sr, ground_truth
 
 
# ============================================================
# CELL 8 — STREAM-LEVEL SLIDING-WINDOW DETECTION
# ============================================================
 
WINDOW_SECONDS = 10
HOP_SECONDS = 5
 
 
def ground_truth_at(ground_truth, t_start, t_end):
    mid = (t_start + t_end) / 2
    for ev in ground_truth:
        if ev["start_sec"] <= mid <= ev["end_sec"]:
            return ev
    return None
 
 
def run_stream_level_test(broadcast, sr, ground_truth, condition, registered_ids):
    degraded = degrade_samples(broadcast, sr, condition)
 
    win = int(WINDOW_SECONDS * sr)
    hop = int(HOP_SECONDS * sr)
 
    tp = fp = tn = fn = 0
    fp_latencies, wm_bers = [], []
 
    for start in range(0, len(degraded) - win, hop):
        chunk = degraded[start:start + win]
        t0, t1 = start / sr, (start + win) / sr
        truth = ground_truth_at(ground_truth, t0, t1)
 
        t_a = time.perf_counter()
        fp_match, _ = fingerprint_recognize(chunk, sr)
        fp_latencies.append(time.perf_counter() - t_a)
 
        wm_result = watermark_detect(chunk, registered_ids=registered_ids)
        if wm_result["ber"] is not None:
            wm_bers.append(wm_result["ber"])
 
        # a window counts as "detected" if EITHER engine identifies the true track
        detected_id = fp_match or wm_result["identified_id"]
        truth_id = truth["track_id"] if truth else None
 
        if truth_id and detected_id == truth_id:
            tp += 1
        elif truth_id and detected_id != truth_id:
            fn += 1
        elif (not truth_id) and detected_id:
            fp += 1
        else:
            tn += 1
 
    total = tp + fp + tn + fn
    return {
        "condition": condition,
        "accuracy": (tp + tn) / total if total else None,
        "tpr": tp / (tp + fn) if (tp + fn) else None,
        "fpr": fp / (fp + tn) if (fp + tn) else None,
        "mean_latency_s": float(np.mean(fp_latencies)) if fp_latencies else None,
        "mean_ber": float(np.mean(wm_bers)) if wm_bers else None,
    }
 
 
# ============================================================
# CELL 9 — RUN FULL EXPERIMENT (5 conditions x 2 reliability runs)
# ============================================================
 
import pandas as pd
 
N_RUNS = 2
 
 
def run_full_experiment(metadata):
    print("Running unit-level validation...")
    unit_results = run_unit_level_tests(metadata)
    unit_df = pd.DataFrame(unit_results)
 
    print("Building simulated broadcast...")
    registered_ids = {str(r["custom_artist_id"]) for r in metadata}
    broadcast, sr, ground_truth = build_simulated_broadcast(metadata)
 
    print("Running stream-level tests...")
    stream_rows = []
    for run_idx in range(N_RUNS):
        for condition in DEGRADATION_CONDITIONS:
            result = run_stream_level_test(broadcast, sr, ground_truth, condition, registered_ids)
            result["run"] = run_idx + 1
            stream_rows.append(result)
            print(f"  run {run_idx+1} [{condition}] "
                  f"acc={result['accuracy']:.3f} tpr={result['tpr']:.3f} "
                  f"fpr={result['fpr']:.3f}")
 
    stream_df = pd.DataFrame(stream_rows)
    stream_summary = stream_df.groupby("condition").agg(["mean", "std"])
 
    unit_df.to_csv(_os.path.join(OUTPUT_DIR, "unit_level_results.csv"), index=False)
    stream_df.to_csv(_os.path.join(OUTPUT_DIR, "stream_level_results_raw.csv"), index=False)
    stream_summary.to_csv(_os.path.join(OUTPUT_DIR, "stream_level_results_summary.csv"))
 
    return unit_df, stream_df, stream_summary